# Running AI Models Locally

In [[26056657-3785-482A-8408-C85C447D8D25|Using the OpenAI SDK]] your code sent each request across the internet to a model in a data center. A language model is a very large file of numbers plus a program that runs it, and smaller models can run on an ordinary laptop. That is called running a model **locally**.

Why would you?

- **Privacy.** Your messages never leave your computer.
- **Cost.** There is no API key and no charge per token.
- **Offline.** Once the model is downloaded, it works without an internet connection.

The trade-off is size. A model that fits on a laptop is smaller than the ones in a data center, so it knows less and makes more mistakes, and how fast it writes depends on your computer.

The good news is that almost nothing changes in your code. Local model apps speak the same language as the OpenAI API, so you use the same SDK and change one line: the address.

# Setting Up a Local Server

You need an app that downloads models and runs them. This notebook works with the two most popular ones. Both are free, and both run on Windows, macOS, and Linux. You need a laptop or desktop computer for this, not a phone or a Chromebook.

Each app runs a small **server** on your computer. A server is a program that waits for requests and answers them, just like OpenRouter does, only this one is running on your own machine.

| | LM Studio | Ollama |
|---|---|---|
| Get it from | lmstudio.ai | ollama.com |
| Download a model | Search inside the app | `ollama pull gemma3` in a terminal |
| Start the server | **Developer** tab, then start the server | Starts with the app |
| Address | `http://localhost:1234/v1` | `http://localhost:11434/v1` |

Start with a small model, around 4 GB or less, such as a Gemma or Qwen model with "4b" in its name. Bigger models give better answers but need more memory.

## Allowing Codetto to connect

A local server ignores requests from web pages unless you tell it they are welcome. This safety rule is called **CORS**, and you need to switch it on once.

- **LM Studio:** in the server settings on the Developer tab, turn on **Enable CORS**.
- **Ollama:** quit Ollama, then start it from a terminal with the `OLLAMA_ORIGINS` setting naming this site:

```
OLLAMA_ORIGINS=https://codetto.app ollama serve
```

# Connecting to the Server

The address `localhost` means "this computer". The number after the colon is the **port**, which says which program on the computer should get the request. LM Studio listens on port `1234` and Ollama on `11434`.

Compare this cell with the cloud version. The `base_url` now points at your own computer, and the `api_key` is a placeholder. A local server doesn't check keys, but the SDK insists on being given one.

Python in Codetto runs inside your browser, so `localhost` here is the computer you are sitting at.

Choose your app's address and run the cell. `client.models.list()` asks the server which models it has, which is a quick way to check the connection.

In [ ]:
import openai

BASE_URL = "http://localhost:1234/v1" #@param ["http://localhost:1234/v1", "http://localhost:11434/v1"]

client = openai.OpenAI(base_url=BASE_URL, api_key='not-needed')

try:
  model_names = [model.id for model in client.models.list()]
  print(f'Connected to {BASE_URL}. Models on this computer:')
  for name in model_names:
    print(' ', name)
except openai.APIConnectionError:
  model_names = []
  print(f'Could not reach {BASE_URL}.')
  print('Check that the server is running and that CORS is switched on.')

## If it can't connect

| Check | How to fix it |
|---|---|
| Is the server running? | Open the app and start the server. In LM Studio its status should say *Running*. |
| Is it the right address? | Pick the address for the app you are using. |
| Is CORS on? | See the setup steps above, then restart the server. |
| Did the browser ask for permission? | Some browsers ask before a page may reach programs on your computer or local network. Choose **Allow**. |
| Still stuck? | Try Chrome or Edge. Some browsers stop web pages from reaching `localhost` at all. |

# Generating Text

This cell is the same request you would send to a cloud model. Only the model name is different, because it has to be one of the models on *your* computer.

Leave `MODEL` empty to use the first name in the list above, or type a name from that list. Some entries in the list may not be chat models (anything with "embed" or "image" in its name, for example), so type a name if the first one gives an error.

The first request can take a while, because the server has to load the model into memory. After that it stays loaded and answers faster.

In [ ]:
MODEL = "" #@param

model = MODEL or model_names[0]
print('Using', model)
print()

response = client.chat.completions.create(
  model=model,
  messages=[
    {"role": "system", "content": "You are a friendly science tutor. Answer in three sentences or fewer."},
    {"role": "user", "content": "Why does the Moon change shape during the month?"},
  ]
)

print(response.choices[0].message.content)

# Streaming and Speed

A local model writes at the speed of your computer, so streaming matters even more here. Watching the words appear is much nicer than staring at an empty cell.

It also lets you measure your own machine. This cell streams a reply, counts the pieces as they arrive, and times the whole thing. Each piece is roughly one token, so the result is close to your computer's speed in **tokens per second**. Try a smaller or larger model and compare.

In [ ]:
import time

stream = client.chat.completions.create(
  model=model,
  messages=[
    {"role": "user", "content": "Write a short poem about a laptop that dreams of being a supercomputer."},
  ],
  stream=True
)

pieces = 0
start = time.time()

for chunk in stream:
  if chunk.choices and chunk.choices[0].delta.content:
    print(chunk.choices[0].delta.content, end='', flush=True)
    pieces += 1

seconds = time.time() - start
print()
print()
print(f'{pieces} pieces in {seconds:.1f} seconds, about {pieces / seconds:.0f} per second.')

# Local or Cloud?

Neither is better at everything. It depends on what you are building.

| | Local model | Cloud model |
|---|---|---|
| Where it runs | Your computer | A data center |
| Cost | Free after the download | A small charge per token |
| Privacy | Nothing leaves your computer | Your messages go to the provider |
| Needs internet | No | Yes |
| Quality | Good for simple tasks | The most capable models |
| Speed | Depends on your computer | Usually fast |
| Setup | Install an app, download a model | Get an API key |

Because the SDK is the same, you can build with a local model while you experiment, then change the `base_url` and model name to switch to a cloud model, or the other way round. Everything else in [[26056657-3785-482A-8408-C85C447D8D25|Using the OpenAI SDK]], such as the system role and conversations, works here too.

# Check Your Understanding